In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder , OneHotEncoder , StandardScaler,MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.compose  import ColumnTransformer
from sklearn.linear_model import LinearRegression


data=pd.read_csv("/content/dataset.csv",index_col=False)
# Drop 'Unnamed: 0' column if it exists, as it's likely an artifact from saving/loading CSV
data = data.drop(columns=["Unnamed: 0"], errors='ignore')

def derived_col():
    data["STRIKE RATE"]=(data["RUNS_SCORED"]/data["BALLS_FACED"])*100
    data["BOUNDARY"]=((data["FOURS"]+data["SIXES"])/data["BALLS_FACED"])*100
        # Replace infinite values
    data.replace([float("inf"), -float("inf")], 0, inplace=True)

    # Replace NaN
    data.fillna(0, inplace=True)
    return data
DATA=derived_col()
# print(DATA)

#-------------------------------------------------------------STANDARD SCALE--------------------------------------------------------------

#-------------------------------------------------------------MINMAX SCALER----------------------------------------------------------------

min_scale=MinMaxScaler()
DATA["FOURS"]=min_scale.fit_transform(DATA[["FOURS"]])
DATA["SIXES"]=min_scale.fit_transform(DATA[["SIXES"]])
DATA["BOUNDARY"]=min_scale.fit_transform(DATA[["BOUNDARY"]])
# print(DATA)

#-------------------------------------------------------------Label Encoding----------------------------------------------------------------

le=LabelEncoder()
DATA["format_encoded"]=le.fit_transform(DATA["MATCH_FORMAT"])

#-----------------------------------------------------------One Hot Encoding----------------------------------------------------------------

target = ["PLAYER_NAME", "OPPONENT_TEAM", "VENUE"]

ON_Encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

ct = ColumnTransformer(
    transformers=[("OH_Encoder", ON_Encoder, target)],
    remainder="passthrough"
)

DATA = DATA.drop(columns=["MATCH_FORMAT"])

DATA = ct.fit_transform(DATA)

DATA = pd.DataFrame(DATA, columns=ct.get_feature_names_out())
# print(DATA.head())


# -------------------------------------------------------- FEATURES AND TARGET -----------------------------------------------------------------

X = DATA.drop(columns=[
    "remainder__RUNS_SCORED",
    "remainder__BALLS_FACED",
    "remainder__FOURS",
    "remainder__SIXES"
])

y = DATA[[
    "remainder__RUNS_SCORED",
    "remainder__BALLS_FACED",
    "remainder__FOURS",
    "remainder__SIXES"
]]

# print(X.shape)
# print(y.shape)


#---------------------------------------------------------------------------------Train and test -------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#---------------------------------------------------------------------------------Model training--------------------------

model = LinearRegression()

model.fit(X_train, y_train)

prediction = model.predict(X_test)

# print(prediction[:5])

#--------------------------------------------------------------------Starting user section----------------------------------

player=input("Enter the name of player:")
opponent_team=input("Enter the opponent Team:")
Venue_Ground=input("Enter the Venue/Ground:")
format=input("Enter the game format[T20I/ODI]:")

format_encoded=le.fit_transform([format])

user_input = pd.DataFrame({
    "MATCH_ID":[0],
    "PLAYER_NAME":[player],
    "OPPONENT_TEAM":[opponent_team],
    "VENUE":[Venue_Ground],
    "RUNS_SCORED":[0],
    "BALLS_FACED":[0],
    "FOURS":[0],
    "SIXES":[0],
    "STRIKE RATE":[0],
    "BOUNDARY":[0],
    "format_encoded":[format_encoded]
})


user_encoded = ct.transform(user_input)

# convert to dataframe
user_encoded = pd.DataFrame(user_encoded, columns=ct.get_feature_names_out())

# keep only the columns used during training
user_encoded = user_encoded[X.columns]

prediction = model.predict(user_encoded)

runs = prediction[0][0]
balls = prediction[0][1]
fours = prediction[0][2]
sixes = prediction[0][3]




strike_rate = (runs / balls) * 100 if balls != 0 else 0
boundary_rate = (fours + sixes) / balls if balls != 0 else 0

print("\nPredicted Performance")
print("----------------------")

print("Runs Scored:", round(runs))
print("Balls Faced:", round(balls))
print("Fours:", round(fours))
print("Sixes:", round(sixes))

print("Strike Rate:", round(strike_rate,2))
print("Boundary per ball:", round(boundary_rate,3))